In [88]:
import os
import sys
import random
import math
import ast
import warnings
from pathlib import Path

# Audio processing modules
import soundfile as sf

# Data handling and visualization modules
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from IPython.display import Audio, display, HTML

# Skikit-learn preprocessing modules
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import KFold
from sklearn.model_selection import GroupKFold
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import StratifiedGroupKFold

# PyTorch modules
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchaudio
import torchaudio.transforms as T
import torchvision.models as models
import torchvision.transforms as TV
from torchinfo import summary
#/kaggle/input/competitions/birdclef-2026/

In [89]:
EXAMPLE_AUDIO_PATH = "/kaggle/input/competitions/birdclef-2026/train_audio/1161364/iNat1114648.ogg"

SAMPLE_RATE = 32000

#transform params
N_MELS = 128
N_FFT = 1024
HOP_LENGTH = 320
SAMPLE_DURATION = 5

N_SAMPLES = SAMPLE_RATE * SAMPLE_DURATION   # Number of samples of preprocessed audio files
F_MIN       = 20                            # Minimum frequency
F_MAX       = 16000                         # Maximum frequency
BATCH_SIZE = 64

In [90]:
def generate_audio_slices(audio_full_paths):
    slices =[]
    i = 0
    for audio_full_path in tqdm(audio_full_paths):
        if i == 100:
            break
        info = sf.info(audio_full_path)
        total_samples = info.frames
    
        start_frame = 0
        end_frame = N_SAMPLES

        i += 1
    
        while end_frame < total_samples: #make sure the next frame doesnt go over
            slices.append((audio_full_path, start_frame, end_frame))
            start_frame += N_SAMPLES
            end_frame += N_SAMPLES

    return pd.DataFrame(slices, columns=["full_path", "start_frame", "end_frame"])

In [91]:
trainval = pd.read_csv('/kaggle/input/competitions/birdclef-2026/train.csv')
#print(trainval.head())
#conv trainval start/end sec -> frames
trainval_extra = pd.read_csv('/kaggle/input/competitions/birdclef-2026/train_soundscapes_labels.csv')
trainval_extra["extra_start_frame"] = pd.to_timedelta(trainval_extra['start']).dt.total_seconds().astype(int) * SAMPLE_RATE
trainval_extra["extra_end_frame"] = pd.to_timedelta(trainval_extra['end']).dt.total_seconds().astype(int) * SAMPLE_RATE
trainval_extra.drop(columns=["start","end"], inplace=True)
#print(trainval_extra.head())
taxonomy = pd.read_csv('/kaggle/input/competitions/birdclef-2026/taxonomy.csv')

# Merge and standardize label informations from different columns of data source tables (all labels in a single string seperated by ;)
trainval['all_labels'] = trainval.apply(lambda row: ';'.join([row['primary_label']] + ast.literal_eval(row['secondary_labels'])), axis=1)
trainval_extra = trainval_extra.drop_duplicates().reset_index(drop=True).rename(columns={'primary_label': 'all_labels'})
trainval_extra['primary_label'] = trainval_extra['all_labels'].str.split(';').str[0]
trainval_extra['secondary_labels'] = trainval_extra['all_labels'].str.split(';').str[1:]

# Include full paths in the data source tables
trainval['full_path'] = '/kaggle/input/competitions/birdclef-2026/train_audio/' + trainval['filename']
trainval_extra['full_path'] = '/kaggle/input/competitions/birdclef-2026/train_soundscapes/' + trainval_extra['filename']

# Create a list of 5 sec slices of audio files and merge information into trainval
audio_full_paths = trainval['full_path'].unique().tolist()
slices_df = generate_audio_slices(audio_full_paths)
trainval = pd.merge(trainval, slices_df, on='full_path', how='right')

# Merge the two datasets into one single dataframe
trainval = pd.concat([trainval, trainval_extra], axis=0)
print(trainval.head())

  0%|          | 100/35549 [00:00<03:13, 183.24it/s]

  primary_label secondary_labels type  latitude  longitude scientific_name  \
0       1161364               []   []  -22.7562   -46.8666    Guyalna cuta   
1       1161364               []   []  -22.7562   -46.8666    Guyalna cuta   
2       1161364               []   []  -22.7562   -46.8666    Guyalna cuta   
3       1161364               []   []  -22.7558   -46.8700    Guyalna cuta   
4       1161364               []   []  -22.7558   -46.8700    Guyalna cuta   

    common_name class_name  inat_taxon_id         author  ... rating  \
0  Guyalna cuta    Insecta      1161364.0  Lucas Barbosa  ...    0.0   
1  Guyalna cuta    Insecta      1161364.0  Lucas Barbosa  ...    0.0   
2  Guyalna cuta    Insecta      1161364.0  Lucas Barbosa  ...    0.0   
3  Guyalna cuta    Insecta      1161364.0  Lucas Barbosa  ...    0.0   
4  Guyalna cuta    Insecta      1161364.0  Lucas Barbosa  ...    0.0   

                                                 url                 filename  \
0  https://static

In [92]:
## Fitting multi label binarizer

all_unique_labels = trainval['all_labels'].str.split(';').explode().unique()
mlb = MultiLabelBinarizer()
mlb.fit([all_unique_labels])
print(f"\nTotal unique labels: {len(all_unique_labels)}")
print(f"Label classes: {mlb.classes_}")


Total unique labels: 81
Label classes: ['1161364' '116570' '1176823' '1491113' '1595929' '209233' '22930' '22956'
 '22961' '22967' '22973' '23158' '24279' '24321' '25073' '25092' '326272'
 '43435' '47144' '47158son01' '47158son02' '47158son03' '47158son04'
 '47158son05' '47158son06' '47158son07' '47158son08' '47158son09'
 '47158son10' '47158son11' '47158son12' '47158son13' '47158son14'
 '47158son15' '47158son16' '47158son17' '47158son18' '47158son19'
 '47158son20' '47158son21' '47158son22' '47158son23' '47158son24'
 '47158son25' '516975' '517063' '555146' '65377' '65380' '66971' '67107'
 '67252' '74113' 'bafcur1' 'bufpar' 'bunibi1' 'chacha1' 'chvcon1' 'compau'
 'compot1' 'fusfly1' 'grekis' 'hyamac1' 'limpki' 'litnig1' 'magant1'
 'nacnig1' 'orwpar' 'plcjay1' 'purjay1' 'redjun' 'rufhor2' 'ruther1'
 'rutjac1' 'sibtan2' 'strher2' 'thlwre1' 'trsowl' 'undtin1' 'wfwduc1'
 'whtdov']


In [93]:
class AnimalSoundDataset(Dataset):
    def __init__(self, dataframe, mlb=mlb, submissioning=False):
        self.dataframe = dataframe
        self.mlb = mlb
        self.sample_rate = SAMPLE_RATE
        self.max_audio_samples = N_SAMPLES
        self.submissioning = submissioning
        
    def __len__(self):
        return len(self.dataframe)

    def _load_audio_slice_at(self, path, start_sample, end_sample):
        stereo_waveform, sr = sf.read(path, start=start_sample, stop=end_sample, dtype='float32', always_2d=True)
        waveform = stereo_waveform.mean(axis=1)
        waveform = torch.from_numpy(waveform).unsqueeze(0)
        return waveform, sr

    def _process_audio(self, audio_path_full, start_samples):
        # Cut 5 sec audio sample with defined starting time from given audio file
        waveform, sr = self._load_audio_slice_at(audio_path_full, start_samples)

        # Resample waveform if necessary
        if sr != self.sample_rate:
            resampler = T.Resample(orig_freq=sr, new_freq=self.sample_rate)
            waveform = resampler(waveform)

        # Padding or truncating sample if necessary
        current_n_samples = waveform.shape[1]
        if current_n_samples < self.max_audio_samples:
            padding_needed = self.max_audio_samples - current_n_samples
            waveform = F.pad(waveform, (0, padding_needed))
        elif current_n_samples > self.max_audio_samples:
            waveform = waveform[:, :self.max_audio_samples]
        return waveform

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        audio_path_full = row['full_path']
        start_samples = row['start_sample']
        end_samples = row['end_sample']
        labels = row['all_labels'].split(';')
        row_id = row['row_id']

        # Preprocess audio file
        waveform = self._process_audio(audio_path_full, start_samples, end_samples)

        # Label encoding (Multi onehot coding)
        one_hot_labels = torch.tensor(self.mlb.transform([labels]).squeeze(0), dtype=torch.float32)

        return (waveform, one_hot_labels) if not self.submissioning else (waveform, one_hot_labels, row_id)

In [94]:
class MelSpecHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.mel_transform = T.MelSpectrogram(
            sample_rate=SAMPLE_RATE,
            n_fft=N_FFT,        # FFT window size
            hop_length=HOP_LENGTH,    # step size between frames
            n_mels=N_MELS,        # number of mel bands
            f_min=F_MIN,
            f_max=F_MAX,
            power=2.0          # power spectrogram (2.0 = energy)
        )
        self.amp_to_db_transform = T.AmplitudeToDB(top_db=80)
        
    def forward(self, x):
        mel_S_db = self.amp_to_db_transform(self.mel_trans(x))
        smp_min = mel_S_db.amin(dim=(1,2,3), keepdim=True)
        smp_max = mel_S_db.amax(dim=(1,2,3), keepdim=True)
        mel_S_db_norm = (mel_S_db - smp_min) / (smp_max - smp_min + 1e-8)
        return mel_S_db_norm

class ResNetHead(nn.Module):
    def __init__(self):
        super().__init__()\
        #out dim - (b, 512, H, W) where H and W are 1 from the avgpool
        resnet18 = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        resnet18.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False) #adapt to spectrogram
        resnet_modules = list(resnet18.children())[:-1] #output fc
        self.backbone = nn.Sequential(*resnet_modules)

    def forward(self, x):
        return self.backbone(x)

class Classifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.melspec = MelSpecHead()
        self.resnet = ResNetHead()
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512,512),
            nn.GELU(),
            nn.Linear(512,num_classes)
        )
        
    def forward(self,x):
        x = self.melspec(x)
        x = self.resnet(x)
        x = self.classifier(x)
        return x     

In [95]:
summary(Classifier(243))

Layer (type:depth-idx)                             Param #
Classifier                                         --
├─MelSpecHead: 1-1                                 --
│    └─MelSpectrogram: 2-1                         --
│    │    └─Spectrogram: 3-1                       --
│    │    └─MelScale: 3-2                          --
│    └─AmplitudeToDB: 2-2                          --
├─ResNetHead: 1-2                                  --
│    └─Sequential: 2-3                             --
│    │    └─Conv2d: 3-3                            3,136
│    │    └─BatchNorm2d: 3-4                       128
│    │    └─ReLU: 3-5                              --
│    │    └─MaxPool2d: 3-6                         --
│    │    └─Sequential: 3-7                        147,968
│    │    └─Sequential: 3-8                        525,568
│    │    └─Sequential: 3-9                        2,099,712
│    │    └─Sequential: 3-10                       8,393,728
│    │    └─AdaptiveAvgPool2d: 3-11              

In [ ]:
class MultiLabelLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, logits, targets):
        bce_loss = F.binary_cross_entropy_with_logits(logits, targets, weight=None, pos_weight=None, reduction='mean') #sigmoid already done internally
        return bce_loss